# Four ways to benchmark wrong
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week6/Benchmarking_Rules_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 6, Module 4: Benchmarking and the cost model**  
**Author:** Dr. Kaoutar El Maghraoui

A benchmark answers how long something takes. A profile answers where the time goes, and the profiler itself adds overhead. Compiled code needs extra rules: warm up first so that compilation happens outside the timed part, synchronize before reading the clock because device calls return before the device finishes, time many steady steps, and report the median. Report the cold compile time, the first warm call, the steady-state median and peak memory separately.

The lecture adds a gate: a benchmark on a compiled target is valid only if no compilation ran inside the timed window. Divide the slowest iteration by the fastest. Above about 1.1, reject the run.

In this notebook you time the same compiled model in four wrong ways and one right way, then plant a recompile inside the timed window and let the spread check catch it.

**What you will do**
1. Compile a small model.
2. Time it with compilation inside the window, without synchronization, and without warm-up.
3. Time it with the profiler running.
4. Time it correctly and report the four numbers the lecture asks for.
5. Plant a recompile in the timed window and apply the spread check.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. The notebook takes about a minute.

## 1. The model
A stack of four linear layers with GELU, in FP16, on a batch of 64 rows. It is compiled with `dynamic=False`, as a fixed-shape accelerator would be.

In [ ]:
import statistics
import time
import torch
import torch.nn as nn

assert torch.cuda.is_available(), "This notebook needs a GPU: Runtime > Change runtime type > T4 GPU"
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

torch.manual_seed(0)
model = nn.Sequential(*[m for _ in range(4) for m in (nn.Linear(2048, 2048), nn.GELU())]).cuda().half().eval()
x = torch.randn(64, 2048, device="cuda", dtype=torch.float16)


def fresh_compile():
    torch._dynamo.reset()
    return torch.compile(model, dynamic=False)

## 2. Wrong: compilation inside the timed window
The first call to a compiled function compiles it. Timing that call measures the compiler, not the model.

In [ ]:
compiled = fresh_compile()
with torch.inference_mode():
    t0 = time.perf_counter()
    compiled(x)
    torch.cuda.synchronize()
    cold = (time.perf_counter() - t0) * 1e3
print(f"first call, including compilation: {cold:,.0f} ms")

## 3. Wrong: no synchronization
GPU calls return as soon as the work is queued. Without `torch.cuda.synchronize()` before reading the clock, the timer measures how fast the CPU can queue work.

In [ ]:
with torch.inference_mode():
    for _ in range(10):
        compiled(x)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(100):
        compiled(x)
    no_sync = (time.perf_counter() - t0) / 100 * 1e3      # the clock is read before the GPU finishes
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(100):
        compiled(x)
    torch.cuda.synchronize()
    with_sync = (time.perf_counter() - t0) / 100 * 1e3
print(f"without synchronize: {no_sync:.3f} ms per call")
print(f"with synchronize:    {with_sync:.3f} ms per call")

**Reading the output**
- Without synchronization the number is smaller, sometimes much smaller. It is the time to queue the work, not to run it. The queued work is still running when the clock is read.

## 4. Wrong: no warm-up
Even after compilation, the first calls can be slower: memory is allocated for the first time, and kernels are loaded. The cell recompiles and times the first ten calls one by one.

In [ ]:
compiled = fresh_compile()
times = []
with torch.inference_mode():
    for _ in range(10):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        compiled(x)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1e3)
print("first ten calls (ms):", " ".join(f"{t:.2f}" for t in times))
print(f"mean of the ten: {statistics.mean(times):.2f} ms, median: {statistics.median(times):.3f} ms")

**Reading the output**
- The first call includes compilation, and the second can still be slower than the rest.
- A mean over these calls is dominated by the first one. A median is robust to it, but only warm-up removes it.

## 5. Wrong: the profiler is running
The profiler records every operator and kernel. That costs time, more for small operations. Use it to find where to look, never to report a speedup.

In [ ]:
from torch.profiler import profile, ProfilerActivity


def timed_calls(n=50):
    with torch.inference_mode():
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n):
            compiled(x)
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1e3


off = timed_calls()
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, with_stack=True):
    on = timed_calls()
print(f"profiler off: {off:.3f} ms per call")
print(f"profiler on:  {on:.3f} ms per call ({on / off:.2f}x)")

## 6. Right: warm up, synchronize, repeat, report the median
The cell follows the lecture's rules and reports the four numbers separately. The spread check divides the slowest timed iteration by the fastest.

In [ ]:
def benchmark(fn, warmup=10, iters=100):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        times = []
        for _ in range(iters):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            fn()
            torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1e3)
    return times


compiled = fresh_compile()
torch.cuda.reset_peak_memory_stats()
with torch.inference_mode():
    t0 = time.perf_counter(); compiled(x); torch.cuda.synchronize()
    cold_ms = (time.perf_counter() - t0) * 1e3
    t0 = time.perf_counter(); compiled(x); torch.cuda.synchronize()
    first_warm_ms = (time.perf_counter() - t0) * 1e3
times = benchmark(lambda: compiled(x))
spread = max(times) / min(times)

print(f"cold compile:        {cold_ms:9,.0f} ms")
print(f"first warm call:     {first_warm_ms:9.3f} ms")
print(f"steady-state median: {statistics.median(times):9.3f} ms  (p10 {statistics.quantiles(times, n=10)[0]:.3f}, p90 {statistics.quantiles(times, n=10)[-1]:.3f})")
print(f"peak memory:         {torch.cuda.max_memory_allocated() / 1e6:9.0f} MB")
print(f"spread, slowest / fastest: {spread:.2f} -> {'accept' if spread <= 1.1 else 'check the slow iterations'}")

**Reading the output**
- The steady-state median is the number to report as the time per call. The cold compile and first warm call are separate lines: on a compiled-schedule machine compile time can be long, and it deserves its own entry.
- The cold compile here is much shorter than in section 2. Inductor reused the kernels it compiled earlier from its on-disk cache. To report a true cold compile, start from an empty cache, for example with `torch._inductor.config.force_disable_caches = True`.
- On a shared Colab GPU, a few iterations can be slow because of other activity, so the spread may sit a little above 1.1 even in a clean run. The next section shows what a real problem looks like.

## 7. A recompile inside the timed window
The cell plants a problem from the lecture: halfway through the timed loop, the input shape changes. With `dynamic=False`, that call compiles a new graph, inside the timed window.

In [ ]:
x_other = torch.randn(32, 2048, device="cuda", dtype=torch.float16)
calls = {"n": 0}


def sometimes_new_shape():
    calls["n"] += 1
    return compiled(x_other if calls["n"] == 60 else x)   # call 60 has a new shape


times = benchmark(sometimes_new_shape, warmup=10, iters=100)
spread = max(times) / min(times)
print(f"median {statistics.median(times):.3f} ms, mean {statistics.mean(times):.3f} ms, slowest {max(times):,.1f} ms")
print(f"spread, slowest / fastest: {spread:,.0f} -> {'accept' if spread <= 1.1 else 'REJECT: something compiled or warmed up inside the window'}")

**Reading the output**
- One iteration takes as long as a compile. The median hardly moves, so a summary that reports only the median would hide it. The mean jumps to many times the median, but without the per-iteration times you could not tell why.
- The spread is in the hundreds or thousands, far above 1.1, and the gate rejects the run. This is why the lecture says to check the spread before trusting a number.
- The fix is to use the same shapes in warm-up and timing, or to precompile every shape at warm-up, which the bucket demo in this folder shows.

## Try this
1. In section 6, set `warmup=0`. Does the spread check catch it?
2. In section 5, turn off `with_stack` and `record_shapes`. How much of the profiler's overhead goes away?
3. Benchmark the eager model, `model(x)`, with the same function, and compute the speedup from compilation.

## Summary
- Do not time compilation, do not read the clock without synchronizing, and do not average in the warm-up calls.
- Do not report numbers measured with the profiler running.
- Report four numbers: cold compile, first warm call, steady-state median, peak memory.
- Check the spread between the slowest and fastest iteration. A large spread means something compiled or warmed up inside the window.

Lecture: Week 6, Module 4, slides 45 and 46 (benchmarking compiled code, no compile in the timed window).